# 00. Start: trzy plany napromieniania

Notatnik pokazuje, jak korzystać z pakietu `offtarget`, i odtwarza ilustrację z prezentacji 23.10.

Model: $\partial_t u = D\,\partial_{xx} u + \rho u (1 - u/K) - \beta R(x) U(t) u$, jednostki bezwymiarowe ($D = \rho = K = 1$).

**Uwaga:** dawki są tu ilustracyjne. $\beta = 1{,}5$ znaczy, że jedna frakcja zostawia $e^{-1{,}5} \approx 22\%$ komórek;
klinicznie (2 Gy) to raczej $\beta \approx 0{,}12$. Zakresy kliniczne ustala Zadanie 2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import Params, Schedule, beam, grid
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.solvers import simulate

rng = seed_everything()

Siatka: obszar o długości 80 (40 szerokości frontu w każdą stronę), 400 przedziałów.
Harmonogram: 10 frakcji co 0,4 jednostki czasu, od $t = 1$.

In [ ]:
x, dx = grid(80.0, 400)
t = np.linspace(0.0, 10.0, 201)
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)

plans = {
    "szeroka wiązka, duża dawka": Params(beta=1.5, s=0.0, r=20.0),
    "szeroka wiązka, mała dawka": Params(beta=0.45, s=0.0, r=20.0),
    "wiązka przesunięta": Params(beta=1.5, s=6.0, r=8.0),
}
runs = {name: simulate(p, schedule, x, t) for name, p in plans.items()}

Obserwable: masa guza $M(t) = \int u\,dx$, najmniejsza masa po leczeniu i czas, po którym masa wraca do $M(0)$.

In [ ]:
rows = []
for name, u in runs.items():
    M = tumor_mass(u, x)
    m_min, t_min = min_mass(t, M)
    rows.append({"plan": name, "M_min / M(0)": m_min / M[0], "t_min": t_min,
                 "czas do odrostu": time_to_regrowth(t, M)})
table = pd.DataFrame(rows)
table.to_csv(TABLES / "00_scenariusze.csv", index=False)
table

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for name, u in runs.items():
    M = tumor_mass(u, x)
    a1.plot(t, M / M[0], lw=2, label=name)
a1.axvspan(schedule.times[0] - 0.1, schedule.times[-1] + 0.1, color="0.9", lw=0, label="frakcje")
a1.set(xlabel="czas t", ylabel="M(t) / M(0)", title="Masa guza dla trzech planów", ylim=(0, None))
a1.legend(fontsize=8)

u = runs["wiązka przesunięta"]
p = plans["wiązka przesunięta"]
a2.fill_between(x, 0, beam(x, p), color="tab:orange", alpha=0.15, label="pole wiązki R(x)")
for k, lab in [(0, "t = 0"), (100, "t = 5, po leczeniu"), (200, "t = 10, odrost")]:
    a2.plot(x, u[k], lw=2, label=lab)
a2.set(xlabel="x", ylabel="gęstość u", title="Wiązka przesunięta: odrost z nienapromienionego brzegu",
       xlim=(-30, 30))
a2.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "00_scenariusze.png", dpi=150)